In [2]:
pip install simpy

In [3]:
"""
===========================================================
SIMULACIÓN DE UNA AGENCIA BANCARIA - DOS COLAS
Horario de cajeros:
  - Cajero 1: 9:00 am - 6:00 pm (almuerzo 1:00 - 2:00 pm)
  - Cajero 2: 9:00 am - 6:00 pm (almuerzo 2:00 - 3:00 pm)
  - Cajero 3: 1:00 pm - 6:00 pm (turno corto, sin almuerzo)
===========================================================
"""
import simpy
import random
import math

# ==========================================================
# VARIABLES DE ENTRADA
# ==========================================================

HORA_INICIO = 9
HORA_CIERRE = 18
HORARIO_ATENCION = (HORA_CIERRE - HORA_INICIO) * 60   # 540 minutos

# Capacidad máxima de la cola/sala INTERNA
CAPACIDAD_COLA_INTERNA = 6

# --- Llegadas: EXPONENCIAL ---
TIEMPO_ENTRE_LLEGADAS = 2.88

# --- Atención: LOGNORMAL ---
TIEMPO_ATENCION_MEDIA = 5.848
TIEMPO_ATENCION_DESV  = 4.044

_var = TIEMPO_ATENCION_DESV ** 2
_sigma2 = math.log(1 + (_var / (TIEMPO_ATENCION_MEDIA ** 2)))
SIGMA_LOGNORMAL = math.sqrt(_sigma2)
MU_LOGNORMAL = math.log(TIEMPO_ATENCION_MEDIA) - (_sigma2 / 2)

rng_llegadas = random.Random(50)
rng_atencion = random.Random(51)

# ==========================================================
# HORARIO DE CAJEROS (en minutos desde HORA_INICIO)
# ==========================================================
EVENTOS_CAJEROS = [
    (0,   +1, "Cajero 1 inicia turno (9:00 am)"),
    (0,   +1, "Cajero 2 inicia turno (9:00 am)"),
    (240, -1, "Cajero 1 sale a almorzar (1:00 pm)"),
    (240, +1, "Cajero 3 inicia turno de refuerzo (1:00 pm)"),
    (300, +1, "Cajero 1 vuelve de almorzar (2:00 pm)"),
    (300, -1, "Cajero 2 sale a almorzar (2:00 pm)"),
    (360, +1, "Cajero 2 vuelve de almorzar (3:00 pm)"),
]
# Capacidad inicial en t=0 (antes de aplicar el primer evento)
CAPACIDAD_INICIAL_CAJEROS = 0

# ==========================================================
# VARIABLES DE ESTADO
# ==========================================================

longitud_cola_externa = 0
longitud_cola_interna = 0

# ==========================================================
# VARIABLES DE SALIDA
# ==========================================================

numero_clientes_llegaron = 0
numero_clientes_atendidos = 0
numero_clientes_perdidos = 0

lista_espera_externa = []
lista_espera_interna = []
lista_tiempo_atencion = []
lista_tiempo_sistema = []

# ==========================================================
# VARIABLES AUXILIARES (áreas bajo la curva)
# ==========================================================

area_cola_externa = 0.0
area_cola_interna = 0.0
ultimo_evento_externa = 0.0
ultimo_evento_interna = 0.0

tiempo_ocupado_total = 0.0

area_capacidad_cajeros = 0.0
ultimo_evento_capacidad = 0.0
capacidad_actual_cajeros = CAPACIDAD_INICIAL_CAJEROS


# ==========================================================
# CONVERSIÓN DE MINUTOS A HORA REAL
# ==========================================================

def hora_real(minutos_simulacion):
    total_minutos = int(HORA_INICIO * 60 + minutos_simulacion)
    horas = total_minutos // 60
    minutos = total_minutos % 60
    return f"{horas:02d}:{minutos:02d}"


# ==========================================================
# ACTUALIZAR ÁREAS
# ==========================================================

def actualizar_area_externa(env):
    global area_cola_externa, ultimo_evento_externa
    tiempo = env.now - ultimo_evento_externa
    area_cola_externa += longitud_cola_externa * tiempo
    ultimo_evento_externa = env.now

def actualizar_area_interna(env):
    global area_cola_interna, ultimo_evento_interna
    tiempo = env.now - ultimo_evento_interna
    area_cola_interna += longitud_cola_interna * tiempo
    ultimo_evento_interna = env.now

def actualizar_area_capacidad(env):
    global area_capacidad_cajeros, ultimo_evento_capacidad
    tiempo = env.now - ultimo_evento_capacidad
    area_capacidad_cajeros += capacidad_actual_cajeros * tiempo
    ultimo_evento_capacidad = env.now


# ==========================================================
# PROCESO: HORARIO DE CAJEROS (turnos + almuerzos)
# ==========================================================

def controlar_horario_cajeros(env, cajeros):
    """Aplica los cambios de capacidad de cajeros según los turnos
    y horarios de almuerzo definidos en EVENTOS_CAJEROS."""

    global capacidad_actual_cajeros

    tiempo_actual = 0
    for minuto_evento, delta, descripcion in EVENTOS_CAJEROS:
        espera = minuto_evento - tiempo_actual
        if espera > 0:
            yield env.timeout(espera)
        tiempo_actual = minuto_evento

        actualizar_area_capacidad(env)
        cajeros._capacity += delta
        capacidad_actual_cajeros = cajeros._capacity

        print(f"{hora_real(env.now)} -> {descripcion} "
              f"(cajeros activos: {cajeros._capacity})")

        if delta > 0:
            # Si aumentó la capacidad, revisa la cola de espera de cajero
            cajeros._trigger_put(None)


# ==========================================================
# PROCESO CLIENTE
# ==========================================================

def cliente(env, nombre, sala_interna, cajeros):

    global numero_clientes_atendidos
    global numero_clientes_perdidos
    global tiempo_ocupado_total
    global longitud_cola_externa
    global longitud_cola_interna

    llegada = env.now

    print(f"{hora_real(env.now)} -> {nombre} llega y entra a la cola EXTERNA")

    actualizar_area_externa(env)
    longitud_cola_externa += 1

    tiempo_restante = max(0, HORARIO_ATENCION - env.now)

    with sala_interna.request() as solicitud_interna:

        resultado = yield solicitud_interna | env.timeout(tiempo_restante)

        actualizar_area_externa(env)
        longitud_cola_externa -= 1

        if solicitud_interna not in resultado:
            numero_clientes_perdidos += 1
            print(
                f"{hora_real(env.now)} -> {nombre} se RETIRA sin ser atendido "
                f"(cierre; seguía en la cola externa)"
            )
            return

        entrada_interna = env.now
        espera_externa = entrada_interna - llegada
        lista_espera_externa.append(espera_externa)

        actualizar_area_interna(env)
        longitud_cola_interna += 1

        print(f"{hora_real(env.now)} -> {nombre} entra a la cola INTERNA")

        with cajeros.request() as solicitud_cajero:

            yield solicitud_cajero

            actualizar_area_interna(env)
            longitud_cola_interna -= 1

            inicio_atencion = env.now
            espera_interna = inicio_atencion - entrada_interna
            lista_espera_interna.append(espera_interna)

            print(f"{hora_real(env.now)} -> {nombre} inicia atención")

            servicio = rng_atencion.lognormvariate(MU_LOGNORMAL, SIGMA_LOGNORMAL)
            lista_tiempo_atencion.append(servicio)
            tiempo_ocupado_total += servicio

            yield env.timeout(servicio)

            numero_clientes_atendidos += 1
            lista_tiempo_sistema.append(env.now - llegada)

            print(f"{hora_real(env.now)} -> {nombre} sale")


# ==========================================================
# GENERADOR DE CLIENTES
# ==========================================================

def llegadas(env, sala_interna, cajeros):

    global numero_clientes_llegaron
    contador = 1

    while True:
        tiempo = rng_llegadas.expovariate(1 / TIEMPO_ENTRE_LLEGADAS)

        if env.now + tiempo >= HORARIO_ATENCION:
            break

        yield env.timeout(tiempo)

        numero_clientes_llegaron += 1

        env.process(cliente(env, f"Cliente {contador}", sala_interna, cajeros))

        contador += 1


# ==========================================================
# EJECUCIÓN
# ==========================================================

env = simpy.Environment()

sala_interna = simpy.Resource(env, capacity=CAPACIDAD_COLA_INTERNA)

# Cajeros arranca en 0 y el proceso de horario le va sumando/restando
# capacidad según los turnos y almuerzos definidos arriba
cajeros = simpy.Resource(env, capacity=CAPACIDAD_INICIAL_CAJEROS or 1)
cajeros._capacity = CAPACIDAD_INICIAL_CAJEROS  # forzar arranque en 0

env.process(llegadas(env, sala_interna, cajeros))
env.process(controlar_horario_cajeros(env, cajeros))

env.run()

tiempo_total_operacion = env.now
minutos_extra = max(0, tiempo_total_operacion - HORARIO_ATENCION)

actualizar_area_capacidad(env)


# ==========================================================
# INDICADORES DE DESEMPEÑO
# ==========================================================

print("\n")
print("="*65)
print("INDICADORES DE DESEMPEÑO")
print("="*65)

promedio_espera_externa = (
    sum(lista_espera_externa) / len(lista_espera_externa)
) if lista_espera_externa else 0

promedio_espera_interna = (
    sum(lista_espera_interna) / len(lista_espera_interna)
) if lista_espera_interna else 0

promedio_atencion = (
    sum(lista_tiempo_atencion) / len(lista_tiempo_atencion)
) if lista_tiempo_atencion else 0

promedio_sistema = (
    sum(lista_tiempo_sistema) / len(lista_tiempo_sistema)
) if lista_tiempo_sistema else 0

cola_externa_promedio = area_cola_externa / tiempo_total_operacion
cola_interna_promedio = area_cola_interna / tiempo_total_operacion

utilizacion = (
    tiempo_ocupado_total / area_capacidad_cajeros
) * 100 if area_capacidad_cajeros > 0 else 0

promedio_cajeros_activos = area_capacidad_cajeros / tiempo_total_operacion

clientes_hora = numero_clientes_atendidos / (tiempo_total_operacion / 60)

print(f"Horario oficial                  : {HORA_INICIO}:00 - {HORA_CIERRE}:00")
print("Turnos de cajeros:")
print("  Cajero 1: 9:00 am - 6:00 pm (almuerzo 1:00 - 2:00 pm)")
print("  Cajero 2: 9:00 am - 6:00 pm (almuerzo 2:00 - 3:00 pm)")
print("  Cajero 3: 1:00 pm - 6:00 pm (sin almuerzo)")
print(f"Capacidad de la cola interna     : {CAPACIDAD_COLA_INTERNA} personas")
print(f"Clientes que llegaron            : {numero_clientes_llegaron}")
print(f"Clientes atendidos               : {numero_clientes_atendidos}")
print(f"Clientes perdidos (cierre)       : {numero_clientes_perdidos}")
print(f"Tiempo promedio espera EXTERNA   : {promedio_espera_externa:.2f} min")
print(f"Tiempo promedio espera INTERNA   : {promedio_espera_interna:.2f} min")
print(f"Tiempo promedio de atención      : {promedio_atencion:.2f} min")
print(f"Tiempo promedio en el sistema     : {promedio_sistema:.2f} min")
print(f"Longitud promedio cola EXTERNA   : {cola_externa_promedio:.2f} personas")
print(f"Longitud promedio cola INTERNA   : {cola_interna_promedio:.2f} personas "
      f"(máx. {CAPACIDAD_COLA_INTERNA})")
print(f"Promedio de cajeros activos      : {promedio_cajeros_activos:.2f}")
print(f"Utilización de cajeros           : {utilizacion:.2f}%")
print(f"Clientes atendidos por hora      : {clientes_hora:.2f}")
print(f"Hora real del último cliente     : {hora_real(tiempo_total_operacion)}")
print(f"Minutos extra tras el cierre     : {minutos_extra:.2f} min")

print("="*65)

09:00 -> Cajero 1 inicia turno (9:00 am) (cajeros activos: 1)
09:00 -> Cajero 2 inicia turno (9:00 am) (cajeros activos: 2)
09:01 -> Cliente 1 llega y entra a la cola EXTERNA
09:01 -> Cliente 1 entra a la cola INTERNA
09:01 -> Cliente 1 inicia atención
09:02 -> Cliente 2 llega y entra a la cola EXTERNA
09:02 -> Cliente 2 entra a la cola INTERNA
09:02 -> Cliente 2 inicia atención
09:04 -> Cliente 1 sale
09:05 -> Cliente 2 sale
09:05 -> Cliente 3 llega y entra a la cola EXTERNA
09:05 -> Cliente 3 entra a la cola INTERNA
09:05 -> Cliente 3 inicia atención
09:06 -> Cliente 4 llega y entra a la cola EXTERNA
09:06 -> Cliente 4 entra a la cola INTERNA
09:06 -> Cliente 4 inicia atención
09:08 -> Cliente 5 llega y entra a la cola EXTERNA
09:08 -> Cliente 5 entra a la cola INTERNA
09:12 -> Cliente 4 sale
09:12 -> Cliente 5 inicia atención
09:16 -> Cliente 3 sale
09:17 -> Cliente 5 sale
09:18 -> Cliente 6 llega y entra a la cola EXTERNA
09:18 -> Cliente 6 entra a la cola INTERNA
09:18 -> Cliente 